In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.model_selection import train_test_split

In [2]:
X,y = load_diabetes(return_X_y = True)

In [3]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=.2, random_state=2)

In [4]:
print(f"Shape X_train -> {X_train.shape}")
print(f"Shape X_test -> {X_test.shape}")
print(f"Shape y_train -> {y_train.shape}")
print(f"Shape y_test -> {y_test.shape}")

Shape X_train -> (353, 10)
Shape X_test -> (89, 10)
Shape y_train -> (353,)
Shape y_test -> (89,)


In [8]:
lr = LinearRegression()
lr.fit(X_train,y_train)
y_pred_lr = lr.predict(X_test)

In [9]:
print(f"Intercept -> {lr.intercept_}")
print(f"Coefficients -> {lr.coef_}")

Intercept -> 151.88331005254167
Coefficients -> [  -9.15865318 -205.45432163  516.69374454  340.61999905 -895.5520019
  561.22067904  153.89310954  126.73139688  861.12700152   52.42112238]


In [10]:
print(f"R2 Score -> {r2_score(y_test,y_pred_lr)}")
print(f"MAE -> {mean_absolute_error(y_test,y_pred_lr)}")
print(f"RMSE -> {root_mean_squared_error(y_test,y_pred_lr)}")

R2 Score -> 0.4399338661568968
MAE -> 45.213034190469024
RMSE -> 55.627840795469155


In [80]:
# Making a class MiniBatchGDRegressor
import random
class MBGDRegressor:
    def __init__(self,lr = 0.01, epochs = 100 , batch_size = 35):
        self.coef_ = None
        self.intercept_ = None
        self.batch_size = batch_size
        self.epochs = epochs
        self.learning_rate = lr

    def fit(self,X_train,y_train):
        self.intercept_ = 0
        self.coef_ = np.ones(X_train.shape[1])
        
        for i in range(self.epochs):

            for j in range(int(X_train.shape[0]/ self.batch_size)):
                idx = random.sample(range(X_train.shape[0]),self.batch_size)
                
                # Finding intercept
                y_hat = self.intercept_ + np.dot(X_train[idx] , self.coef_)
                intercept_der = -2 * np.mean(y_train[idx] - y_hat)
                self.intercept_ = self.intercept_ - self.learning_rate * intercept_der

                # Finding Coefficients 
                coef_der = -2 * np.dot((y_train[idx] - y_hat),X_train[idx])/ len(idx)
                self.coef_ = self.coef_ - self.learning_rate * coef_der


    def predict(self,X_test):
        return self.intercept_ + np.dot(X_test , self.coef_)

In [103]:
mbgdr = MBGDRegressor(batch_size=int(X_train.shape[0]/10), lr = 0.35 , epochs = 50)
mbgdr.fit(X_train,y_train)
print(f"Intercept -> {mbgdr.intercept_}")
print(f"Coefficients -> {mbgdr.coef_}")

Intercept -> 154.06397046683617
Coefficients -> [  55.22885438  -67.64253022  356.77968727  246.74537322   17.42366244
  -27.49729869 -176.84345085  133.44190425  321.60028877  124.25278475]


In [104]:
y_pred_mbgdr = mbgdr.predict(X_test)

In [105]:
print(f"R2 Score -> {r2_score(y_test, y_pred_mbgdr)}")

R2 Score -> 0.4340779394315347


In [122]:

# We can apply Mini Batch Gradient Descent using scikit-learn SGDRegressor() but there is a trick

# We have to use partial_fit() function which always has epoch i.e max_iter  = 1
gdr = SGDRegressor(loss="squared_error" , learning_rate = "constant" ,eta0 = 0.2)

In [123]:
batch_size = 35
idx = random.sample(range(X_train.shape[0]) , batch_size)

for i in range(50):
    gdr.partial_fit(X_train[idx] , y_train[idx])

In [124]:
print(gdr.coef_)
print(gdr.intercept_)

[  22.16700731  -19.60563995  496.09124187  256.47202973  -11.25088058
 -156.47231335  -99.10909867  -39.27398141  460.4007737    94.84473632]
[143.64278874]


In [125]:
y_pred_gdr = gdr.predict(X_test)

In [126]:
print(r2_score(y_test, y_pred_gdr))

0.385782010150033
